# Forecast — corrida inicial

Se corre **una sola vez** (o cuando quieras rehacer todo desde cero).

Backtestea los últimos `backtest_horizon` meses cerrados con todos los modelos, elige el mejor por
serie y proyecta `horizon` meses. Escribe el resultado en la tabla de salida, que **ya tiene que
existir**.

Toda la configuración —conexión, tablas, SQL, parámetros del forecast— está en **`fc_oracle.py`**.
Este notebook no tiene nada configurable adentro salvo la celda de parámetros.

### Cómo queda el nodo en el pipeline (Elyra / OpenShift AI)

| Propiedad del nodo | Valor |
|---|---|
| **File Dependencies** | `fc_oracle.py`, `forecast_engine.py` |
| **Environment Variables** | origen: `ORA_USER`, `ORA_PASSWORD`, `ORA_DSN` · destino: `ORA_DEST_USER`, `ORA_DEST_PASSWORD`, `ORA_DEST_DSN` · opcionales: `FC_CUTOFF`, `FC_REESCRIBIR_DESDE`, `FC_DRY_RUN`, `FC_MAX_WORKERS` |
| **Runtime Image** | una con `pandas statsmodels joblib psutil oracledb prophet` |
| **CPU / RAM** | lo que le asignes: el motor lee el límite del pod (cgroups) y ajusta la cantidad de procesos solo |

La contraseña va como *secret* montado en variable de entorno, nunca en el notebook.
Si una celda lanza excepción, el nodo queda en **failed** — que es lo que querés que pase.

In [ ]:
# Celda de parámetros (tag `parameters`): Elyra los pasa por variable de entorno,
# papermill puede inyectarlos acá directamente.
import os

CUTOFF = os.getenv("FC_CUTOFF") or None        # "2026-07-01" para fijar el último mes cerrado
DRY_RUN = os.getenv("FC_DRY_RUN", "0") == "1"  # 1 = calcula y no escribe

# Desde dónde reescribir la tabla de salida:
#   "auto"  -> desde el corte (borra las proyecciones viejas, deja la historia)
#   "todo"  -> borra todo y reinserta
#   6       -> 6 meses antes del corte
#   fecha   -> "2025-01-01"
REESCRIBIR_DESDE = os.getenv("FC_REESCRIBIR_DESDE") or "todo"

print(f"CUTOFF={CUTOFF!r}  REESCRIBIR_DESDE={REESCRIBIR_DESDE!r}  DRY_RUN={DRY_RUN}")

## 1. Setup

In [ ]:
import sys, time
sys.path.insert(0, os.getcwd())   # fc_oracle.py y forecast_engine.py viajan como File Dependencies

import pandas as pd
import fc_oracle as io
from forecast_engine import (PanelForecaster, available_models,
                             available_cpus, available_memory_gb, container_cpu_limit)

log = io.configurar_logging("forecast-inicial")
cfg = io.build_config(cutoff=CUTOFF)

log.info("corte=%s | horizonte=%d | backtest=%d | refit_step=%d",
         cfg.resolved_cutoff().date(), cfg.horizon, cfg.backtest_horizon, cfg.refit_step)
log.info("recursos: %d cpus usables (límite del pod: %s) | %.1f GB de RAM disponibles",
         available_cpus(), container_cpu_limit() or "sin límite", available_memory_gb() or -1)
log.info("modelos: %s", ", ".join(available_models(cfg.models)))
t_inicio = time.time()

## 2. Tabla de salida

Una fila por (categorías, mes) y **una columna por modelo**, más la métrica real y el valor del
ganador. Es la misma tabla que la corrida mensual vuelve a leer como entrada.

Los nombres salen de `COLUMNAS_MODELO` y `COLUMNAS_FIJAS` en `fc_oracle.py`. La celda de abajo
imprime el DDL exacto que corresponde a esa configuración (no lo ejecuta) y valida contra el
diccionario de datos que no falte ninguna columna, **antes** de ponerse a calcular.

In [ ]:
print(io.ddl_sugerido(cfg))

with io.conexion_destino() as conn:
    io.validar_tabla(conn, cfg)     # lanza excepción con el ALTER TABLE si falta algo

## 3. Leer la fuente

In [ ]:
with io.conexion_origen() as conn:
    src = io.leer_fuente(conn, cfg)

if src.empty:
    raise RuntimeError("La fuente no devolvió filas: revisá SQL_FUENTE en fc_oracle.py")
src.head()

## 4. Correr el forecast

Acá se va el tiempo de la corrida: `series × orígenes × modelos`. Con `backtest_horizon=12` y
`refit_step=3` son 4 orígenes de backtest + 1 de futuro por serie y modelo.

### Los modelos y por qué están

Todos se reentrenan en cada origen del backtest, compiten con el mismo error y ganan por serie.
Cada uno cubre un patrón distinto: la idea no es que uno sea "el bueno", sino que haya un
candidato razonable para cada forma que puede tener una serie.

| Modelo | Columna | Qué captura | Por qué está |
|---|---|---|---|
| `snaive` | `F_SNAIVE` | mismo mes del año anterior | Piso de comparación obligado en datos con estacionalidad anual. Si un modelo caro no le gana, no justifica su costo. |
| `drift` | `F_DRIFT` | último valor + pendiente media | Referencia de tendencia pura, sin estacionalidad. Barato y sorprendentemente difícil de batir en series cortas. |
| `seasonal_ols` | `F_SEASONAL_OLS` | tendencia lineal + dummy por mes | Regresión cerrada, sin optimización iterativa: nunca falla ni deja de converger. Muy bueno cuando el patrón es estable y limpio. |
| `ses` | `F_SES` | nivel suavizado | Series sin tendencia ni estacionalidad, con ruido. |
| `holt` | `F_HOLT` | nivel + tendencia amortiguada | El amortiguamiento evita que la tendencia se dispare a 12 meses, que es el error clásico de proyectar largo. |
| `ets` | `F_ETS` | Holt-Winters completo | Prueba 6 configuraciones (con/sin tendencia, estacionalidad aditiva o multiplicativa) y elige por AICc. Es el caballo de batalla en datos mensuales de venta y suele ganar la mayoría de las series. |
| `theta` | `F_THETA` | descomposición Theta | Ganó la competencia M3 y sigue siendo top-5 en las M4/M5. Rendimiento altísimo para lo que cuesta. |
| `stl_ets` | `F_STL_ETS` | STL robusto + ETS en la tendencia | STL aísla la estacionalidad resistiendo outliers. Es el que mejor aguanta meses atípicos (una promo, un mes con un cliente raro) sin deformar el patrón. |
| `sarima` | `F_SARIMA` | ARIMA estacional | Cubre autocorrelaciones que los suavizados no modelan. Grilla acotada de 6 órdenes elegida por AICc en vez de búsqueda exhaustiva, porque es el segundo más caro. |
| `prophet` | `F_PROPHET` | tendencia por tramos + estacionalidad Fourier | Es el único que absorbe cambios de nivel/pendiente (un cliente que cambia de escala) sin ayuda. Bueno con huecos y series irregulares. |
| `croston_sba` | `F_CROSTON_SBA` | demanda intermitente | Para series con muchos ceros, todos los modelos anteriores predicen mal. Croston separa "cuánto" de "cada cuánto"; la variante SBA corrige el sesgo al alza del Croston clásico. |
| `tsb` | `F_TSB` | intermitente con probabilidad | Como Croston pero actualiza la probabilidad de venta en cada período, no sólo cuando hay venta: detecta clientes que se están apagando. |
| `combo_median` | `F_COMBO` | mediana de todos los anteriores | Combinar es la técnica más consistente que existe en forecasting; la mediana además ignora al modelo que se fue al descuido. Suele quedar entre los mejores sin ganar casi nunca — su valor es que **nunca es terrible**. |

Quedaron afuera a propósito: `naive` (dominado por `snaive` en datos mensuales), `mean` (dominado
por `ses`) y `auto_arima` de pmdarima (redunda con `sarima` y suma una dependencia que arrastra
compilación). Están en el registro del motor si los querés: `models=[...]` en `build_config()`.

**Cómo se elige el ganador**, por serie, sobre los `backtest_horizon` meses validados:

```
score = wMAPE ponderado por recencia  +  0.25 × |sesgo acumulado| / nivel
```

El wMAPE mide el error; los pesos decaen a la mitad cada 12 meses (el mes más viejo de la ventana
pesa la mitad que el último), así pesa más el comportamiento reciente; y el término de sesgo
castiga al modelo que se queda sistemáticamente corto o largo, que es lo que pasa cuando no le
pega a la tendencia. Se configura con `metric`, `bias_weight` y `recency_half_life`.

In [ ]:
fc = PanelForecaster(cfg)
out = fc.run(src)

## 5. Control de la corrida

In [ ]:
io.resumen(fc, out, cfg)
fc.summary()      # ranking de modelos: series ganadas y error mediano

In [ ]:
# Vista rápida para dejar evidencia en el notebook ejecutado que archiva el pipeline
serie = out[cfg.category_cols[0]].iloc[0]
columnas = ([cfg.date_col, cfg.actual_col]
            + sorted(c for c in out.columns if c.startswith(cfg.model_col_prefix))
            + [cfg.best_model_col, cfg.forecast_col, cfg.future_flag_col])
out.loc[out[cfg.category_cols[0]] == serie, columnas]

## 6. Guardar

MERGE idempotente por `(categorías, fecha)` con un solo commit: si el nodo reintenta, pisa en vez
de duplicar. Cada fila lleva todas las columnas de modelo de una.

In [ ]:
if DRY_RUN:
    log.warning("DRY_RUN activo: no se escribe nada en %s", io.TABLA_SALIDA)
else:
    with io.conexion_destino() as conn:
        io.guardar(conn, out, cfg, reescribir_desde=REESCRIBIR_DESDE)

log.info("corrida inicial OK en %.1fs", time.time() - t_inicio)